# The Privileged Access Hypothesis

**Survey Reference:** Section 6.3 - The Privileged Access Hypothesis

## Overview

The **Privileged Access Hypothesis** addresses when and why certain bases (like individual neurons) align with interpretable features. Not all directions in activation space are equally "accessible" to the network's computation.

### Key Ideas

- **Privileged bases:** Some coordinate systems are special (e.g., the neuron basis after ReLU)
- **Computational access:** The network can only easily compute on certain representations
- **Implications for interpretability:** We should focus on privileged directions

## Learning Objectives

1. Understand what makes a basis "privileged"
2. Identify privileged bases in transformer components
3. Analyze how nonlinearities create privileged directions
4. Connect to SAE dictionary learning

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. What is a Privileged Basis?

A **privileged basis** is a coordinate system that the network's computation naturally aligns with.

### Examples in Transformers:

| Component | Privileged Basis | Why? |
|-----------|-----------------|------|
| MLP neurons | Neuron basis | ReLU/GELU operates element-wise |
| Residual stream | No privileged basis | Linear transformations are rotation-invariant |
| Attention heads | Head basis | Each head computes independently |
| Token position | Position basis | Causal attention mask operates per-position |

In [ ]:
# Demonstrate: ReLU creates a privileged basis
def show_relu_privilege():
    """Show that ReLU makes the neuron basis special."""
    
    # Random rotation matrix
    d = 10
    Q, _ = torch.linalg.qr(torch.randn(d, d))
    
    # Input in standard basis
    x = torch.randn(100, d)
    
    # Apply ReLU in standard basis
    y1 = torch.relu(x)
    
    # Rotate, apply ReLU, rotate back
    y2 = (torch.relu(x @ Q) @ Q.T)
    
    # These are NOT the same! ReLU is not rotation-invariant
    print(f"Are outputs equal? {torch.allclose(y1, y2)}")
    print(f"Max difference: {(y1 - y2).abs().max():.4f}")

show_relu_privilege()

## 2. The Residual Stream Has No Privileged Basis

The residual stream in transformers is interesting because:
- It's only ever transformed linearly (by attention and MLP projections)
- Any rotation of the residual stream could be absorbed into the weights
- Therefore, there's no "natural" coordinate system

In [ ]:
# Demonstrate rotation invariance of linear operations
def show_residual_stream_no_privilege():
    """The residual stream has no privileged basis."""
    d = 64
    Q, _ = torch.linalg.qr(torch.randn(d, d))
    
    # Original linear layer
    W = torch.randn(d, d)
    b = torch.randn(d)
    
    # Rotated version (absorb rotation into weights)
    W_rot = Q.T @ W @ Q
    b_rot = Q.T @ b
    
    # These compute the same function (up to the rotation)
    x = torch.randn(10, d)
    y1 = x @ W + b
    y2 = (x @ Q) @ W_rot + b_rot
    y2_unrotated = y2 @ Q.T
    
    print(f"Outputs match: {torch.allclose(y1, y2_unrotated, atol=1e-5)}")

show_residual_stream_no_privilege()

## 3. When Do Neurons Become Interpretable?

Neurons become interpretable when:
1. They're in a privileged basis (after nonlinearity)
2. There's no superposition (enough capacity)
3. The network "chooses" to use that neuron for one concept

In [ ]:
# TODO: Analyze monosemantic vs polysemantic neurons in a real model
def analyze_neuron_interpretability(model, layer_idx, neuron_idx):
    """Check if a neuron is monosemantic by analyzing its activation patterns."""
    pass

## 4. Implications for Sparse Autoencoders

SAEs work by creating a new privileged basis:
- The SAE dictionary provides feature directions
- ReLU (or similar) creates privilege for these directions
- Sparsity encourages features to be "chosen" individually

In [ ]:
# TODO: Demonstrate how SAE creates interpretable privilege

## 5. Identifying Privileged Directions Empirically

### Activation Statistics

Privileged directions often show:
- Sparse activation patterns
- Clear on/off behavior
- Semantic coherence in max-activating examples

In [ ]:
# TODO: Compute activation statistics to identify privileged directions
def activation_sparsity(activations):
    """Measure sparsity of activations along each direction."""
    pass

def activation_kurtosis(activations):
    """Kurtosis indicates heavy-tailed (sparse) distributions."""
    pass

## Key Takeaways

1. **Privileged bases** arise from nonlinearities and architectural choices
2. **The residual stream** has no privileged basis - any rotation is valid
3. **Neurons after ReLU** are in a privileged basis - element-wise operations matter
4. **SAEs create privilege** by defining a dictionary with sparse activations
5. **Interpretability is easier** when we work in privileged bases

## Next Steps

→ **04_universality_platonic_representation.ipynb**: Do different models learn the same representations?